# triton #11322 main 검증 노트북

**할 일은 딱 하나**: 런타임 → 런타임 유형 변경 → CPU, 고RAM 체크 → 저장 →
Run all. 그 다음 완료될 때까지(약 30~60분 예상) 기다렸다가
**파일 → GitHub에 사본 저장**을 눌러주세요(브랜치: `colab-11322-verify`,
같은 브랜치로 덮어쓰기).

이 노트북은 전 셀이 무인 실행되도록 작성되어 있습니다 — 중간에 입력할
것 없습니다.


In [ ]:
# 1. 빌드 의존성 설치
import subprocess, sys
pkgs = ["setuptools>=40.8.0", "cmake>=3.20,<4.0", "ninja>=1.11.1", "nanobind==2.10.2"]
r = subprocess.run([sys.executable, "-m", "pip", "install"] + pkgs, capture_output=True, text=True)
print(r.stdout[-1000:])
print(r.stderr[-500:])
print("RC:", r.returncode)


In [ ]:
# 2. clone (cwd 자멸 방지: cd 먼저, rm -rf는 그 다음)
%cd /content
!rm -rf /content/triton
!git clone --branch colab-11322-verify https://github.com/alexxony/triton.git /content/triton
%cd /content/triton
!git log --oneline -3


In [ ]:
# 3. pip triton 있으면 제거 (충돌 방지)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "triton"], capture_output=True, text=True)
print("uninstall done")


In [ ]:
# 4. 본체 빌드 (MAX_JOBS=4 — 고RAM 세션이므로 병렬도 올림)
import subprocess, os, time
env = os.environ.copy()
env["TRITON_BUILD_WITH_CCACHE"] = "0"
env["MAX_JOBS"] = "4"
t0 = time.time()
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-v", "-e", ".", "--no-build-isolation"],
    cwd="/content/triton", env=env, capture_output=True, text=True,
)
elapsed = time.time() - t0
with open("/content/build_full.log", "w") as f:
    f.write(r.stdout)
    f.write(r.stderr)
print(f"RC: {r.returncode}  elapsed: {elapsed:.0f}s")
print("--- tail ---")
print((r.stdout + r.stderr)[-3000:])


In [ ]:
# 5. 빌드 확인
import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import triton; print(triton.__file__); print(triton.__version__)"],
                    capture_output=True, text=True)
print(r.stdout, r.stderr)
BUILD_OK = ("triton" in r.stdout and "/content/triton" in r.stdout)
print("BUILD_OK:", BUILD_OK)


In [ ]:
# 6. #11322 리프로듀서 실행 (compile-only, GPU 불필요)
import triton
import triton.language as tl
from triton.backends.compiler import GPUTarget

@triton.jit
def repro_dot_dominance(
    switch_ptr, a, b, c, M, N, K, stride_am, stride_bn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    values = tl.load(switch_ptr + tl.arange(0, 2))
    zero, one = tl.split(values)
    cond = zero > one
    m_offsets = tl.arange(0, BLOCK_M)
    k_offsets = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_K), tl.float32)
    for n in tl.range(tl.cdiv(N, BLOCK_N)):
        n_offsets = n * BLOCK_N + tl.arange(0, BLOCK_N)
        a_mask = (m_offsets[:, None] < M) & (n_offsets[None, :] < N)
        a_ptrs = a + m_offsets[:, None] * stride_am + n_offsets[None, :]
        a_block = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b_mask = (n_offsets[:, None] < N) & (k_offsets[None, :] < K)
        if cond:
            b_ptrs = b + n_offsets[:, None] * stride_bn + (one * k_offsets)[None, :]
        else:
            b_ptrs = b + n_offsets[:, None] * stride_bn + k_offsets[None, :]
        b_block = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc = tl.dot(a_block, b_block, acc, allow_tf32=False)
    c_mask = (m_offsets[:, None] < M) & (k_offsets[None, :] < K)
    c_ptrs = c + m_offsets[:, None] * K + k_offsets[None, :]
    tl.store(c_ptrs, acc, mask=c_mask)

target = GPUTarget("hip", "gfx942", 64)
src = triton.compiler.ASTSource(
    fn=repro_dot_dominance,
    signature={
        "switch_ptr": "*i32", "a": "*fp32", "b": "*fp32", "c": "*fp32",
        "M": "i32", "N": "i32", "K": "i32",
        "stride_am": "i32", "stride_bn": "i32",
        "BLOCK_M": "constexpr", "BLOCK_N": "constexpr", "BLOCK_K": "constexpr",
    },
    constexprs={"BLOCK_M": 32, "BLOCK_N": 32, "BLOCK_K": 32},
)

RESULT = {}
try:
    k = triton.compile(src, target=target, options={"num_stages": 2, "num_warps": 4})
    RESULT["status"] = "COMPILE_SUCCEEDED"
    ttgir = k.asm.get("ttgir", "")
    RESULT["pingpong_markers"] = {
        m: (m in ttgir) for m in ["setprio", "cond_barrier", "sched.barrier"]
    }
except Exception as e:
    RESULT["status"] = "COMPILE_FAILED"
    RESULT["error_type"] = type(e).__name__
    RESULT["error_msg"] = str(e)[:2000]

print("=== #11322 main 검증 결과 ===")
for k_, v_ in RESULT.items():
    print(f"{k_}: {v_}")


## 완료 후 할 일

위 셀 6의 출력(`=== #11322 main 검증 결과 ===` 이하)을 확인한 뒤,
**파일 → GitHub에 사본 저장** → 브랜치 `colab-11322-verify` 선택 →
저장. 이게 안 눌리면 결과가 fork에 반영되지 않습니다.
